# 🔮 World Cup Match Oracle

Welcome to the **Match Prediction Oracle**! This interactive notebook allows you to select any two qualified national teams and instantly predict their matchup probabilities using our trained XGBoost model.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Stylize visual theme
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

# Load model and intermediate matchup features
model_path = os.path.join('..', 'models', 'xgboost_wm_modelV4.joblib')
matchup_path = os.path.join('..', 'data', 'processed', 'model_input', 'matchup_features.csv')

if not os.path.exists(model_path) or not os.path.exists(matchup_path):
    raise FileNotFoundError("Please run the ML pipeline ('python run_pipeline.py all') first to compile model and matchup features.")

model = joblib.load(model_path)
matchup_df = pd.read_csv(matchup_path)

print("🔮 The World Cup Match Oracle is initialized and ready!")

## Define Oracle Core Functionality

We implement the prediction and visualization routine. The oracle searches for the engineered pairwise matchups of interest. If the requested team order is reversed, it multiplies the delta features by `-1` before running prediction to ensure exact symmetrical consistency.

In [ ]:
def predict_match_visual(team_a, team_b):
    """Predicts matchup probabilities between two countries and displays a polished horizontal bar chart."""
    # Search for matching pairwise record in database
    match = matchup_df[(matchup_df['Team_A'].str.lower() == team_a.lower()) & 
                       (matchup_df['Team_B'].str.lower() == team_b.lower())]
    reversed_match = False
    
    if match.empty:
        match = matchup_df[(matchup_df['Team_A'].str.lower() == team_b.lower()) & 
                           (matchup_df['Team_B'].str.lower() == team_a.lower())]
        reversed_match = True
        
    if match.empty:
        print(f"⚠️ Error: No matchup records found for '{team_a}' or '{team_b}'.")
        return
        
    features = [
        'Delta_Total_Market_Value', 'Delta_Median_Top11_Value', 'Delta_Chemistry',
        'Delta_Form_Rating', 'Delta_UCL_Minutes', 'Delta_Tournament_Minutes',
        'Delta_Average_Age', 'Delta_TM_Value_Rank', 'Delta_FIFA_Rank', 'Delta_FIFA_Points'
    ]
    
    X_pred = match[features].copy()
    if reversed_match:
        X_pred = X_pred * -1
        
    # Append 'Is_Neutral' to features as we are simulating neutral grounds for international cup matches
    X_pred['Is_Neutral'] = 1
    
    # Match features list strictly to training requirements
    X_pred = X_pred[features + ['Is_Neutral']]
    
    probs = model.predict_proba(X_pred)[0]
    
    # Plot the predicted probability distribution
    outcomes = [f"Win {team_b}", "Draw", f"Win {team_a}"]
    colors = ['#f8766d', '#619cff', '#00ba38']
    
    plt.figure(figsize=(9, 4.5))
    bars = plt.barh(outcomes, probs, color=colors, edgecolor='grey', height=0.55)
    plt.xlim(0, 1.05)
    plt.title(f"🎯 Match Prognosis: {team_a} vs. {team_b}", fontsize=14, pad=15, fontweight='bold')
    plt.xlabel("Outcome Probability", fontsize=11, labelpad=10)
    
    # Add text labels on the bars
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f"{width*100:.1f}%", 
                 va='center', ha='left', fontweight='bold', fontsize=10)
        
    plt.tight_layout()
    plt.show()

    # Print text summary
    winner_idx = probs.argmax()
    if winner_idx == 2:
        print(f"🏆 Oracle Prediction: {team_a} is favored to WIN!")
    elif winner_idx == 0:
        print(f"🏆 Oracle Prediction: {team_b} is favored to WIN!")
    else:
        print("⚖️ Oracle Prediction: High likelihood of a DRAW / Overtime.")

## Run Match Predictions

Run the cells below with your favorite matchups (e.g., `Germany`, `France`, `Argentina`, `Brazil`, `Switzerland`, `Portugal`, `Spain`, `Morocco`).

In [ ]:
predict_match_visual('Germany', 'France')

In [ ]:
predict_match_visual('Brazil', 'Argentina')

In [ ]:
predict_match_visual('Switzerland', 'Spain')